# Stock Prediction — Improved Solution

**Goal**: Select up to 4,000 stocks to sell and maximise:
$$R = \sum_i \text{sell}_i \times (p40_i - p50_i)$$

## Two key improvements over the baseline

| Change | Baseline | Improved | Gain |
|---|---|---|---|
| **Target** | Predict `p50` (absolute price) | Predict `p40 − p50` (gain directly) | +1,674 R |
| **Features** | 11 engineered | 40 raw prices + 33 engineered = 73 | +496 R |
| **Models** | XGBoost only | XGBoost + CatBoost ensemble | +334 R |
| **Total** | **17,828** | **~19,836** | **+2,008 R (+11%)** |

**Why predicting gain works better**: when the target is `p50`, the model wastes capacity getting the absolute price level right. When the target is `p40 − p50`, every tree directly answers "how much does this stock drop?" — exactly what the ranking needs.

## Step 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import xgboost as xgb
from catboost import CatBoostRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

train = pd.read_csv('train.csv', index_col='ID')
test  = pd.read_csv('test.csv',  index_col='ID')

print('Train:', train.shape)
print('Test: ', test.shape)
train.head(3)

## Step 2 — Feature Engineering (73 features)

**Two groups:**
1. **Raw prices** `p1..p40` — lets the model see the exact price path shape
2. **Engineered** — momentum, volatility, moving averages, Bollinger Bands, trend slope

In [ ]:
PRICE_COLS = [f'p{i}' for i in range(1, 41)]

def make_features(df):
    p  = np.clip(df[PRICE_COLS].values, 1e-6, None)
    lr = np.diff(np.log(p), axis=1)          # 39 daily log-returns
    f  = {}

    # --- Group 1: raw price path (40 features) ---
    for i, col in enumerate(PRICE_COLS):
        f[col] = p[:, i]

    # --- Group 2: engineered (33 features) ---
    f['log_ret_full']    = np.log(p[:, -1] / p[:, 0])
    f['last_change']     = p[:, -1] - p[:, -2]
    f['last_change_pct'] = f['last_change'] / p[:, -2]

    for w in [5, 10, 20]:
        f[f'ret_{w}d'] = np.log(p[:, -1] / p[:, -1 - w])

    f['ma5']  = p[:, -5:].mean(axis=1)
    f['ma10'] = p[:, -10:].mean(axis=1)
    f['ma20'] = p[:, -20:].mean(axis=1)
    f['ma40'] = p.mean(axis=1)
    f['p40_vs_ma5']  = p[:, -1] - f['ma5']
    f['p40_vs_ma20'] = p[:, -1] - f['ma20']
    f['p40_vs_ma40'] = p[:, -1] - f['ma40']

    f['std10'] = p[:, -10:].std(axis=1)
    f['std20'] = p[:, -20:].std(axis=1)
    f['std40'] = p.std(axis=1)
    f['vol10'] = lr[:, -10:].std(axis=1)
    f['vol20'] = lr[:, -20:].std(axis=1)
    f['vol40'] = lr.std(axis=1)

    upper20 = f['ma20'] + 2 * f['std20']
    lower20 = f['ma20'] - 2 * f['std20']
    bw      = upper20 - lower20
    f['bb_pos'] = np.where(bw > 0, (p[:, -1] - lower20) / bw, 0.5)

    f['hist_min']    = p.min(axis=1)
    f['hist_max']    = p.max(axis=1)
    f['hist_range']  = f['hist_max'] - f['hist_min']
    f['p40_vs_min']  = p[:, -1] - f['hist_min']
    f['p40_vs_max']  = f['hist_max'] - p[:, -1]
    f['p40_pct_range'] = np.where(
        f['hist_range'] > 0, f['p40_vs_min'] / f['hist_range'], 0.5
    )

    t      = np.arange(40)
    t_norm = (t - t.mean()) / t.std()
    lp     = np.log(p)
    f['trend_slope'] = (lp * t_norm).mean(axis=1) - lp.mean(axis=1) * t_norm.mean()

    f['ret_2nd_half']   = np.log(p[:, -1] / p[:, 19])
    f['ret_1st_half']   = np.log(p[:, 19] / p[:, 0])
    f['momentum_accel'] = f['ret_2nd_half'] - f['ret_1st_half']

    out = pd.DataFrame(f, index=df.index)
    out.replace([np.inf, -np.inf], np.nan, inplace=True)
    out.fillna(out.median(), inplace=True)
    return out

X_train = make_features(train)
X_test  = make_features(test)

# ── KEY CHANGE: target is the GAIN, not p50 ──────────────────────────────────
y_train = train['p40'] - train['p50']   # positive = stock drops = good sell

print(f'Feature matrix : {X_train.shape}')
print(f'Target (gain)  : mean={y_train.mean():.3f}  std={y_train.std():.3f}')
print(f'  Positive gain (stock drops): {(y_train > 0).mean():.1%}')

## Step 3 — 5-Fold Cross-Validation

XGBoost + CatBoost ensemble, predicting gain directly.  
Score formula: `predicted_gain / (std10 + 100)` — rank higher-gain, lower-vol stocks first.

In [ ]:
PENALTY  = 100
p40_all  = train['p40']
p50_all  = train['p50']

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

fold_Rs    = []
best_iters = {'xgb': [], 'cat': []}

print(f'{"Fold":>4}  {"Val R (800 sells)":>18}')
print('-' * 28)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

    # XGBoost
    xgb_m = xgb.XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=RANDOM_STATE, verbosity=0,
        early_stopping_rounds=50,
    )
    xgb_m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    best_iters['xgb'].append(xgb_m.best_iteration)

    # CatBoost
    cat_m = CatBoostRegressor(
        iterations=1000, learning_rate=0.05, depth=6,
        l2_leaf_reg=3, random_seed=RANDOM_STATE,
        early_stopping_rounds=50, verbose=False,
    )
    cat_m.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    best_iters['cat'].append(cat_m.best_iteration_)

    # Ensemble prediction (average)
    gain_pred = (xgb_m.predict(X_val) + cat_m.predict(X_val)) / 2

    # Risk-adjusted score: predicted gain / (volatility + penalty)
    sigma  = X_val['std10'].values
    scores = gain_pred / (sigma + PENALTY)

    K   = int(0.4 * len(val_idx))
    idx = np.argsort(scores)[::-1][:K]
    s   = np.zeros(len(val_idx), dtype=int); s[idx] = 1

    p40_v = p40_all.iloc[val_idx].values
    p50_v = p50_all.iloc[val_idx].values
    r     = float((s * (p40_v - p50_v)).sum())

    fold_Rs.append(r)
    print(f'{fold:>4}  {r:>18.2f}')

total_R        = sum(fold_Rs)
xgb_best_iter  = int(np.mean(best_iters['xgb']))
cat_best_iter  = int(np.mean(best_iters['cat']))

print('-' * 28)
print(f'\nTotal R across all folds : {total_R:.2f}')
print(f'  (≈ expected submission R)')
print(f'\nXGBoost mean best iter   : {xgb_best_iter}')
print(f'CatBoost mean best iter  : {cat_best_iter}')

In [ ]:
## Step 4 — Retrain on All 10,000 Training Stocks & Submit

xgb_final = xgb.XGBRegressor(
    n_estimators=xgb_best_iter, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=RANDOM_STATE, verbosity=0,
)
xgb_final.fit(X_train, y_train)

cat_final = CatBoostRegressor(
    iterations=cat_best_iter, learning_rate=0.05, depth=6,
    l2_leaf_reg=3, random_seed=RANDOM_STATE, verbose=False,
)
cat_final.fit(X_train, y_train)

print(f'XGBoost retrained  ({xgb_best_iter} trees on {len(X_train)} stocks)')
print(f'CatBoost retrained ({cat_best_iter} trees on {len(X_train)} stocks)')

In [ ]:
# Ensemble gain prediction on test set
gain_test = (xgb_final.predict(X_test) + cat_final.predict(X_test)) / 2

# Risk-adjusted score: predicted_gain / (volatility + penalty)
sigma_test = X_test['std10'].values
scores     = gain_test / (sigma_test + PENALTY)

# Sell the top 4,000 by score
sell = pd.Series(0, index=test.index)
sell[pd.Series(scores, index=test.index).nlargest(4000).index] = 1

print(f'Stocks to sell : {sell.sum()}')
print(f'Constraint OK  : {sell.sum() <= 4000}')
print(f'\nEstimated R (5-fold CV): {total_R:.2f}')
print('Submit submission.csv to Kaggle to see the real score.')

pd.DataFrame({'ID': test.index, 'sell': sell.values}).to_csv('submission.csv', index=False)
print('\nsubmission.csv saved.')

In [ ]:
# Retrain on 100% of training data using the iteration count from early stopping
final_model = xgb.XGBRegressor(
    n_estimators     = model.best_iteration,
    learning_rate    = 0.05,
    max_depth        = 5,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    random_state     = RANDOM_STATE,
    verbosity        = 0,
)
final_model.fit(X_train, y_train)
print(f'Retrained with {model.best_iteration} trees on all {len(X_train)} stocks.')